# Guide d'utilisation du plugin de backtest factoriel

Ce notebook montre un flux volontairement court : choisir les variables → charger les données utiles → calculer une fois le benchmark → tester les signaux unitaires → tester les ajouts incrémentaux → comparer plusieurs recettes composites.

Chaque backtest peut retourner une figure Plotly. Lors de l'export centralisé, les sorties HTML et PNG peuvent être activées séparément. Les figures présentent Top, Worst et Benchmark, ainsi que les ratios Top/Bench, Worst/Bench et Top/Worst. Par défaut, les résultats restent légers : les builders sont libérés après extraction des performances, métriques et positions. Les mesures avant/après sont documentées dans [RAPPORT_OPTIMISATION_BACKTEST.md](RAPPORT_OPTIMISATION_BACKTEST.md).

## Points d'entrée principaux

- `test_unitary_signals()` : teste par défaut level, pct_1, diff_1 et rank_diff_1. Les horizons 3, 6 et 12 sont ajoutés uniquement lorsqu'ils sont nommés explicitement. Une simple liste récupère automatiquement la direction dans le catalogue `LOWER_IS_BETTER`.
- `test_incremental_signals()` : ajoute chaque variable candidate séparément à un score de base. Les dimensions et les poids réellement utilisés proviennent de `candidate_config`.
- `test_composite_signals()` : exécute en une fois plusieurs recettes composites nommées et indépendantes.

Ces trois fonctions retournent toujours `{'screen': ..., 'results': ...}`. `calculate_composite_score()` conserve les colonnes dérivées dans `screen`.

## 1. Charger le plugin

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\Users\jingx\Downloads\公司回测插件")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import BacktestEngine
import func
import factor_config

importlib.reload(BacktestEngine)
importlib.reload(factor_config)
importlib.reload(func)

from func import (
    RECOMMENDED_PERIOD_BREAKPOINTS,
    build_periods_from_breakpoints,
    calculate_benchmark_performance,
    export_backtest_results,
    load_backtest_data,
    plot_performance_comparison,
    prepare_performance_comparison,
    reconstruct_backtest_analysis,
    test_composite_signals,
    test_incremental_signals,
    test_unitary_signals,
)
from factor_config import (
    FACTOR_FAMILIES,
    RAW_VARIABLES as CATALOG_RAW_VARIABLES,
    factor_columns,
    make_signal_dimensions,
    signal_options,
)

print("Plugin chargé")

## 2. Choisir les familles et préparer la configuration

Le catalogue `FACTOR_FAMILIES` classe les colonnes du screen par famille. `factor_columns()` fournit une liste ciblée et `CATALOG_RAW_VARIABLES` contient tout le catalogue. Le benchmark et les paramètres communs du moteur sont définis une seule fois.

La sélection ci-dessous conserve le catalogue par famille, puis copie explicitement trois noms choisis après l'analyse unitaire. Cette petite liste est volontaire : elle rend les recettes composites faciles à lire et à modifier.

In [ ]:
DATA_DIR = PLUGIN_DIR / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
BENCHMARK = "STOXX EUROPE 600"
START_DATE = "2010-01-01"
N_JOBS = 4
PERIOD_BREAKPOINTS = list(RECOMMENDED_PERIOD_BREAKPOINTS)
EXPORT_ROOT = PLUGIN_DIR / "exports"
EXPORT_NAME = "factor_research_run"
EXPORT_DIR = EXPORT_ROOT / EXPORT_NAME
EXPORT_HTML = False
EXPORT_PNG = False

SELECTED_FAMILIES = ("growth", "quality_profitability", "balance_sheet_leverage")
FAMILY_VARIABLES = factor_columns(*SELECTED_FAMILIES)
RAW_VARIABLES = [
    "Revenue 5Y CAGR",
    "FCF Conversion",
    "Net Debt to Ebit",
]
list_noire_path = None

In [ ]:
# Pour tout tester : RAW_VARIABLES = list(CATALOG_RAW_VARIABLES)
family_sizes = {name: len(columns) for name, columns in FACTOR_FAMILIES.items()}
display(pd.Series(family_sizes, name="Nombre de variables"))
display(pd.Series(RAW_VARIABLES, name="Variables sélectionnées"))

## 3. Charger les données et calculer une fois le benchmark

`load_backtest_data()` lit uniquement les variables demandées et les colonnes techniques. `start_date` limite les rendements et `lookback_periods=12` conserve l'historique requis pour les transformations à 1, 3, 6 et 12 périodes. `compact_dtypes=True`, activé par défaut, compacte ISIN, SEDOL, la région et les codes sectoriels sans réduire la précision des variables financières. `calculate_benchmark_performance()` calcule ensuite le benchmark une seule fois. Sa série est injectée explicitement dans tous les tests par `RUN_OPTIONS`. Si `bench_perf` vaut `None`, le moteur calcule encore le benchmark lui-même.

In [ ]:
screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=RAW_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

print(f"screen : {screen.shape}, returns : {returns.shape}")

### Paramètres communs à tous les tests

Le benchmark est calculé avant les signaux. `RUN_OPTIONS` centralise ensuite le benchmark, la date de début, la fréquence de rebalancement, la méthode de remplissage, les périodes et les options de figure. `N_JOBS=4` exécute plusieurs signaux dans des processus distincts ; utilisez `N_JOBS=1` pour revenir exactement au mode séquentiel. Le premier signal construit si nécessaire `MONTHLY_BASE_CACHE`, puis les suivants réutilisent cette base compacte en parallèle. Top et Worst restent réunis dans une même tâche afin de partager leur préparation mensuelle. Les figures sont calculées dans les workers puis affichées dans le notebook, dans l'ordre de la configuration. Sur quatre signaux réels, `n_jobs=4` réduit le temps de 42,8 %. Commencez avec 2 à 4 workers. Avec `N_JOBS > 1`, conservez `retain_builders=False` et exportez le lot après le calcul au lieu de fournir un `save_path` unique. Le cache se réinitialise automatiquement si l'objet `screen` change ; utilisez `monthly_base_cache=None` uniquement pour le désactiver explicitement.

In [ ]:
BENCH_PERF = calculate_benchmark_performance(
    screen=screen, returns=returns, bench=BENCHMARK, start_date=START_DATE
)

MONTHLY_BASE_CACHE = {}

RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": 0.13,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": MONTHLY_BASE_CACHE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": True,
    "build_figure": True,
}

In [ ]:
display(BENCH_PERF.rename("Benchmark").to_frame().tail())
display(pd.Series({key: value for key, value in RUN_OPTIONS.items() if key != "bench_perf"}))

## 4. Tester les signaux unitaires

Sans paramètre `dimensions`, chaque variable est testée en level, pct_1, diff_1 et rank_diff_1. Les anciens noms pct, diff et rank_diff restent des alias d'une période, mais les colonnes, titres et compositions affichent toujours l'horizon `_1`. `rank_diff_N` calcule d'abord le rang orienté de la société dans sa région et son industrie, puis sa variation par rapport à N observations antérieures ; une valeur positive indique toujours une amélioration du rang.

Pour tester d'autres horizons, indiquez-les explicitement, par exemple `dimensions=("pct_3", "diff_6", "rank_diff_12")`. Pour un lot large, `make_signal_dimensions(periods=(1, 3, 6, 12))` construit automatiquement les 13 tests par variable : un level et trois variations sur quatre horizons. Un horizon compte les observations successives de chaque société dans `screen` : avec un screen mensuel, 12 périodes correspondent à 12 mois. Pour dix à quinze variables, désactivez temporairement `show_plot` et `build_figure` dans `RUN_OPTIONS`, puis tracez seulement les résultats retenus après la comparaison.

In [ ]:
UNITARY_DIMENSIONS = make_signal_dimensions(periods=(1,))
# Exemple large :
# RAW_VARIABLES = factor_columns("growth")[:12]
# UNITARY_DIMENSIONS = make_signal_dimensions(periods=(1, 3, 6, 12))

unitary_batch = test_unitary_signals(
    screen=screen,
    returns=returns,
    signal_config=RAW_VARIABLES,
    list_noire_path=list_noire_path,
    dimensions=UNITARY_DIMENSIONS,
    **RUN_OPTIONS,
)
screen = unitary_batch["screen"]
unitary_export = export_backtest_results(
    results={"unitary": unitary_batch},
    output_dir=EXPORT_ROOT,
    export_name=EXPORT_NAME,
    export_html=EXPORT_HTML,
    export_png=EXPORT_PNG,
)

Les colonnes dérivées portent toujours leur horizon, par exemple `Revenue 5Y CAGR__pct_3` ou `Net Debt to Ebit__rank_diff_12`. Elles sont ajoutées au même `screen` et restent disponibles pour une analyse manuelle ultérieure. Les colonnes temporaires `Unitary_*`, utilisées uniquement pour construire les portefeuilles, sont supprimées après chaque test. Utilisez `retain_builders=True` uniquement pour un diagnostic interne exceptionnel.

In [ ]:
derived_columns = [
    column for column in screen.columns
    if ("__over__" in column or "__pct_" in column
        or "__diff_" in column or "__rank_diff_" in column)
]
derived_columns

## 5. Effectuer une analyse incrémentale

Le score de base est d'abord backtesté une fois. Chaque variable candidate est ensuite ajoutée séparément à cette base. Sans suffixe explicite, pct, diff et rank_diff utilisent une période ; utilisez par exemple `pct_3`, `diff_6` ou `rank_diff_12` pour un autre horizon.

In [ ]:
BASELINE_CONFIG = {
    "Revenue 5Y CAGR": signal_options(level=1.0, pct_1=0.5),
}
CANDIDATE_CONFIG = {
    "FCF Conversion": signal_options(level=1.0, diff_3=0.5),
    "Net Debt to Ebit": signal_options(
        higher_is_better=False, level=1.0, rank_diff_6=0.5
    ),
}

incremental_batch = test_incremental_signals(
    screen=screen,
    returns=returns,
    baseline_config=BASELINE_CONFIG,
    candidate_config=CANDIDATE_CONFIG,
    list_noire_path=list_noire_path,
    **RUN_OPTIONS,
)
screen = incremental_batch["screen"]
incremental_export = export_backtest_results(
    results={"incremental": incremental_batch},
    output_dir=EXPORT_ROOT,
    export_name=EXPORT_NAME,
    export_html=EXPORT_HTML,
    export_png=EXPORT_PNG,
)

## 6. Construire et backtester plusieurs scores composites

Après les tests unitaires, définissez chaque recette avec trois ou quatre variables retenues. `signal_options()` accepte les horizons explicites `pct_N`, `diff_N` et `rank_diff_N`. Les anciens paramètres sans suffixe restent des raccourcis pour N=1. `test_composite_signals()` conserve dans chaque résultat le backtest et sa composition réellement utilisée.

In [ ]:
COMPOSITE_CONFIGS = {
    "Croissance mixte": {
        "Revenue 5Y CAGR": signal_options(level=1.0, pct_3=0.5),
        "FCF Conversion": signal_options(level=1.0, diff_6=0.5),
        "Net Debt to Ebit": signal_options(
            higher_is_better=False, level=1.0, rank_diff_12=0.5
        ),
    },
    "Croissance variations": {
        "Revenue 5Y CAGR": signal_options(pct_12=1.0),
        "FCF Conversion": signal_options(diff_3=1.0),
        "Net Debt to Ebit": signal_options(
            higher_is_better=False, level=0.5, rank_diff_6=0.5
        ),
    },
}

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=list_noire_path,
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
composite_export = export_backtest_results(
    results={"composite": composite_batch},
    output_dir=EXPORT_ROOT,
    export_name=EXPORT_NAME,
    export_html=EXPORT_HTML,
    export_png=EXPORT_PNG,
)

## 7. Reconstruire et combiner les performances

Chaque cellule de test exporte immédiatement son lot dans `EXPORT_DIR`. Tant que `EXPORT_NAME` reste inchangé, les nouveaux tests sont ajoutés aux résultats existants et un `test_path` déjà présent est remplacé par sa version la plus récente. Les lots unitaires, incrémentaux et composites peuvent donc être exécutés séparément, dans n'importe quel ordre et même après un redémarrage du kernel.

`reconstruct_backtest_analysis()` lit uniquement le dossier cumulé et restaure les performances, la Composition, tous les scores, les métriques classiques et les métriques par sous-période. `metrics_by_period` réunit `Période totale` et chaque période de rupture dans une seule table test × période. `prepare_performance_comparison()` découvre ensuite automatiquement tous les tests enregistrés, construit les sélections et les ratios Top/Benchmark, Worst/Benchmark et Top/Worst, puis prépare la figure Plotly.

Le dossier contient notamment `backtest_summary.csv`, `signal_composition.csv`, `classic_metrics.csv`, `period_metrics.csv`, `metrics_by_period.csv`, `backtest_registry.json`, ainsi que les sous-dossiers `data/`, `holdings/` et éventuellement `figures/`. Les données sont toujours écrites avant les images.

In [ ]:
analysis_bundle = reconstruct_backtest_analysis(
    export_dir=EXPORT_DIR,
    portfolios=("Top",),
    performance_save_path=EXPORT_DIR / "all_top_performance.csv",
)

all_top_performance = analysis_bundle["performance"]
performance_composition = analysis_bundle["performance_composition"]
backtest_summary = analysis_bundle["summary"]
classic_metrics = analysis_bundle["classic_metrics"]
period_metrics = analysis_bundle["period_metrics"]
metrics_by_period = analysis_bundle["metrics_by_period"]
signal_composition = analysis_bundle["signal_composition"]
comparison_table = backtest_summary

CLASSIC_COLUMNS = [
    "test_type", "test_name", "robust_score",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_information_ratio",
    "top_beta", "top_tracking_error", "top_sortino_ratio",
]
classic_view = comparison_table.loc[:, [
    column for column in CLASSIC_COLUMNS if column in comparison_table.columns
]].sort_values("robust_score", ascending=False)

COMPOSITION_COLUMNS = [
    "display_path", "raw_variable", "dimension",
    "higher_is_better", "source_higher_is_better",
    "weight", "denominator", "raw_variable_total_weight",
    "raw_variable_absolute_weight_share",
]
available_composition_columns = [
    column for column in COMPOSITION_COLUMNS
    if column in performance_composition.columns
]
composite_composition = performance_composition.loc[
    performance_composition["test_type"].eq("composite"),
    available_composition_columns,
]

display(comparison_table)
display(classic_view)
display(signal_composition)
display(composite_composition)
display(classic_metrics)
display(metrics_by_period)
display(all_top_performance)

In [ ]:
comparison_data = prepare_performance_comparison(
    export_dir=EXPORT_DIR,
    save_path=EXPORT_DIR / "selected_performance_comparison.csv",
)

selected_performance = comparison_data["performance"]
selected_ratios = comparison_data["ratios"]
performance_composition = comparison_data["composition"]
PERFORMANCE_SELECTION = comparison_data["performance_selection"]
RATIO_DEFINITIONS = comparison_data["ratio_definitions"]

comparison_figure = plot_performance_comparison(
    performance=selected_performance,
    ratios=selected_ratios,
    benchmark_column="Benchmark",
    title="Comparaison des performances",
    save_path=None,
    show_plot=True,
    rebase=True,
)

## 8. Comparer les différentes périodes

Tous les points d'entrée utilisent un seul paramètre temporel : la liste d'années `period_breakpoints`. La valeur recommandée `[2020, 2022, 2024]` découpe la chronologie en quatre segments : avant 2020, 2020–2021, 2022–2023 et depuis 2024. Le menu déroulant situé en haut de chaque figure Plotly affiche, pour la période choisie, le CAGR de Top, le CAGR actif et le CAGR Top/Worst. Les courbes Top, Worst et Benchmark repartent toujours de 100 au début de la période sélectionnée.

`backtest_summary.csv` contient les principaux champs au format large `period_<période>_<métrique>`. `period_metrics.csv` contient une ligne complète par couple test × période et convient mieux aux filtres et aux tableaux croisés.

In [ ]:
pd.DataFrame(build_periods_from_breakpoints(RUN_OPTIONS["period_breakpoints"]))

In [ ]:
period_table = period_metrics
period_view = period_table[[
    "period_label", "test_type", "test_name",
    "top_cagr", "active_cagr", "top_worst_cagr",
    "top_sharpe_ratio", "top_max_drawdown",
    "active_cagr_rank_global", "active_cagr_rank_within_type",
]]
period_view.sort_values(["period_label", "active_cagr_rank_global"])

In [ ]:
active_cagr_pivot = period_table.pivot_table(
    index=["test_type", "test_name"],
    columns="period_id",
    values="active_cagr",
)
active_cagr_pivot.sort_values(active_cagr_pivot.columns[-1], ascending=False)

Une année de rupture marque le début d'une nouvelle période. Ainsi, `[2009, 2020]` crée trois segments : avant 2009, 2009–2019 et depuis 2020. Les ruptures recommandées dans le code sont 2020, 2022 et 2024. Ajoutez 2009 si les données couvrent aussi les années antérieures à 2009. Une liste vide `[]` conserve uniquement la période totale.

In [ ]:
ALTERNATIVE_BREAKPOINTS = [2009, 2020]
display(pd.DataFrame(build_periods_from_breakpoints(ALTERNATIVE_BREAKPOINTS)))
# Modifiez RUN_OPTIONS["period_breakpoints"] avant les tests pour utiliser cette variante.

## 9. Lire les objets de résultats

Un résultat de backtest standard contient notamment :

```python
result["robust_score"]
result["top_bench_ratio"]
result["top_worst_ratio"]
result["performance"]
result["ratios"]
result["figure"]
result["top_holdings"]
result["worst_holdings"]
result["period_metrics"]
```

Les trois points d'entrée utilisent la même structure. Par exemple :

```python
unitary_batch["results"]["Net Debt to Ebit | level"]["robust_score"]
incremental_batch["results"]["Net Debt to Ebit"]["robust_score"]
composite_batch["results"]["Croissance mixte"]["robust_score"]
```

Le Robust Score nécessite au moins 756 jours de cotation communs. Si l'historique est insuffisant, il renvoie `NaN`.

Le flux interne est maintenant découplé : `calculate_top_vs_bottom_results()` produit d'abord les performances, les ratios et toutes les métriques, puis `plot_top_vs_bottom_results()` construit la figure à partir de cet objet. Pour un traitement en lot limité aux données, utilisez `show_plot=False, build_figure=False` dans n'importe quel point d'entrée ; les comparaisons restent reconstructibles depuis les fichiers performance locaux.